In [83]:
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()   #see on vajalik, et laadida .env failis olevad serveri ligipääs ja salasõna

print(os.getenv('SUPABASE_URL')) 


https://ueweovriwxiipmhsylmg.supabase.co


In [84]:
# Laen alla vajalikud teegid
%pip install supabase python-dotenv pandas plotly


Note: you may need to restart the kernel to use updated packages.


In [85]:
# impordin uusest, enne pip install, siis import
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()  

print(os.getenv('SUPABASE_URL'))  

https://ueweovriwxiipmhsylmg.supabase.co


In [86]:
# testin andmebaasi ligipääsetavust, peaks andma print pd.dataframe peale 5 rida
from supabase import create_client
import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

# Ühendus ja päring
url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_KEY")

supabase = create_client(url, key)
response = supabase.table("sales").select("*").limit(5).execute()

df = pd.DataFrame(response.data)
print(df)

2026-11-16 21:08:33,245 - INFO - HTTP Request: GET https://ueweovriwxiipmhsylmg.supabase.co/rest/v1/sales?select=%2A&limit=5 "HTTP/2 200 OK"


   id  sale_id        invoice_id            sale_date  customer_id  \
0   1        1  INV-202301-00001  2023-01-10T00:00:00         2588   
1   2        2  INV-202301-00002  2023-01-16T00:00:00         4338   
2   3        3  INV-202301-00003  2023-01-05T00:00:00         4673   
3   4        4  INV-202301-00004  2023-01-02T00:00:00         4677   
4   5        5  INV-202301-00005  2023-01-13T00:00:00         2390   

   product_id  quantity  unit_price  total_price channel store_location  \
0        1274         2      234.79       469.58    pood        Tallinn   
1        1207         2      241.13       482.26    pood          Pärnu   
2        1264         1      258.46       221.19    pood          Pärnu   
3        1341         3       45.21       135.63    pood          Tartu   
4        1284         1       99.57        99.57    pood          Tartu   

  payment_method  
0          kaart  
1      järelmaks  
2      järelmaks  
3       sularaha  
4          kaart  


In [87]:
# Kiendiandmete import
from supabase import create_client
from dotenv import load_dotenv
import os, pandas as pd

load_dotenv()
supabase = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))
df_orders = pd.DataFrame(supabase.table('sales').select('*').execute().data)
df_customers = pd.DataFrame(supabase.table('customers').select('*').execute().data)
print(f"Tellimusi: {len(df_orders)}, Kliente: {len(df_customers)}")

2026-11-16 21:08:33,461 - INFO - HTTP Request: GET https://ueweovriwxiipmhsylmg.supabase.co/rest/v1/sales?select=%2A "HTTP/2 200 OK"
2026-11-16 21:08:33,579 - INFO - HTTP Request: GET https://ueweovriwxiipmhsylmg.supabase.co/rest/v1/customers?select=%2A "HTTP/2 200 OK"


Tellimusi: 1000, Kliente: 1000


In [88]:
# 1B Filtreeri ja kombineeri 
# Variant A (Supabase): .select('*').eq('store_location', 'Tallinn').order('total_price', desc=True)
# Variant B (pandas):
df_tallinn = df_orders.merge(df_customers, on='customer_id')
df_tallinn = df_tallinn[df_tallinn['store_location'] == 'Tallinn'].sort_values('total_price', ascending=False)
print(f"Tallinna tellimusi: {len(df_tallinn)}, Käive: {df_tallinn['total_price'].sum():.2f} EUR")

df_Pärnu = df_orders.merge(df_customers, on='customer_id')
df_Pärnu = df_Pärnu[df_Pärnu['store_location'] == 'Pärnu'].sort_values('total_price', ascending=False)
print(f"Pärnu tellimusi: {len(df_Pärnu)}, Käive: {df_Pärnu['total_price'].sum():.2f} EUR")
print(f"Kogukäive: {df_Pärnu['total_price'].sum():.2f} EUR")
print(f"Keskmine tellimus: {df_Pärnu['total_price'].mean():.2f} EUR")

Tallinna tellimusi: 103, Käive: 29759.10 EUR
Pärnu tellimusi: 36, Käive: 10169.34 EUR
Kogukäive: 10169.34 EUR
Keskmine tellimus: 282.48 EUR


In [89]:
# Tellimused ja käive linnade lõike (veebipood on välja arvatud)
def city_report(df, city):
    """Genereeri asukohapõhine müügiraport."""
    city_data = df[df['store_location'] == city]
    return {'store_location': city, 'orders': len(city_data),
            'revenue': city_data['total_price'].sum()}

for city in ['Tallinn', 'Tartu', 'Pärnu']:
    r = city_report(df_orders, city)
    print(f"{r['store_location']}: {r['orders']} tellimust, {r['revenue']:.2f} EUR")


Tallinn: 398 tellimust, 112060.36 EUR
Tartu: 175 tellimust, 51383.15 EUR
Pärnu: 115 tellimust, 30989.46 EUR


In [90]:
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def safe_fetch(supabase_client, table_name):
    """Too andmed turvalise vigade käsitlemisega."""
    try:
        response = supabase_client.table(table_name).select('*').execute()
        df = pd.DataFrame(response.data)
        if len(df) == 0:
            logger.warning(f"Tabel '{table_name}' on tühi!")
        logger.info(f"Laaditud {len(df)} rida tabelist '{table_name}'")
        return df
    except Exception as e:
        logger.error(f"Viga tabeli '{table_name}' lugemisel: {e}")
        return pd.DataFrame()  # Tagasta tühi DataFrame vea korral


In [91]:
# Keskmise tellimuse suurus ja kogukäive 1000 tellimuse puhul
from datetime import datetime

def weekly_sales_report(df, report_date=None):
    """Genereeri iganädalane müügiraport.

    Args:
        df: DataFrame müügitellimustega
        report_date: Raporti kuupäev (vaikimisi täna)
    Returns:
        dict: Raporti kokkuvõte
    """
    if report_date is None:
        report_date = datetime.now().strftime('%Y-%m-%d')
    return {
        'report_date': report_date,
        'total_orders': len(df),
        'total_revenue': round(df['total_price'].sum(), 2),
        'avg_order': round(df['total_price'].mean(), 2),
    }

# Käivita
result = weekly_sales_report(df_orders)
for key, value in result.items():
    print(f"  {key}: {value}")


  report_date: 2026-11-16
  total_orders: 1000
  total_revenue: 285483.53
  avg_order: 285.48


In [92]:
def fetch_all_orders(supabase_client, table_name='sales'):
    """Laeb tabelist kõik read, ületades Supabase 1000 rea limiidi."""
    all_data = []
    batch_size = 1000
    start = 0
    
    while True:
        response = supabase_client.table(table_name).select('*').range(start, start + batch_size - 1).execute()
        batch = response.data
        if not batch:
            break
        all_data.extend(batch)
        if len(batch) < batch_size:
            break
        start += batch_size
        
    return pd.DataFrame(all_data)

# Kasutamine:
df_all_orders = fetch_all_orders(supabase, 'sales')
print(f"Kokku laaditud tellimusi: {len(df_all_orders)}")

2026-11-16 21:08:33,762 - INFO - HTTP Request: GET https://ueweovriwxiipmhsylmg.supabase.co/rest/v1/sales?select=%2A&offset=0&limit=1000 "HTTP/2 200 OK"
2026-11-16 21:08:33,878 - INFO - HTTP Request: GET https://ueweovriwxiipmhsylmg.supabase.co/rest/v1/sales?select=%2A&offset=1000&limit=1000 "HTTP/2 200 OK"
2026-11-16 21:08:33,981 - INFO - HTTP Request: GET https://ueweovriwxiipmhsylmg.supabase.co/rest/v1/sales?select=%2A&offset=2000&limit=1000 "HTTP/2 200 OK"
2026-11-16 21:08:34,068 - INFO - HTTP Request: GET https://ueweovriwxiipmhsylmg.supabase.co/rest/v1/sales?select=%2A&offset=3000&limit=1000 "HTTP/2 200 OK"
2026-11-16 21:08:34,173 - INFO - HTTP Request: GET https://ueweovriwxiipmhsylmg.supabase.co/rest/v1/sales?select=%2A&offset=4000&limit=1000 "HTTP/2 200 OK"
2026-11-16 21:08:34,257 - INFO - HTTP Request: GET https://ueweovriwxiipmhsylmg.supabase.co/rest/v1/sales?select=%2A&offset=5000&limit=1000 "HTTP/2 200 OK"
2026-11-16 21:08:34,362 - INFO - HTTP Request: GET https://ueweovri

Kokku laaditud tellimusi: 10118


In [ ]:
# Arvuta kogukäive kõigi tellimuste puhul   
total_revenue_all = df_all_orders['total_price'].sum()
total_revenue_all = df_all_orders['total_price'].sum()
print(f"Kogukäive kõigi tellimuste puhul: {total_revenue_all:.2f} EUR")

Kogukäive kõigi tellimuste puhul: 2909177.98 EUR


In [ ]:
#Harjutus 2B -parimad ja nõrgimad kliendid RFM analüüsi põhjal. See ei ole sama mis nädal 7 tulemused.
def calculate_rfm(df, reference_date=None):
    """Arvuta RFM skoorid ja segmendid.

    Args:
        df: DataFrame tellimustega (veerud: customer_id, sale_date, total_price)
        reference_date: Viitekuupäev Recency arvutamiseks

    Returns:
        DataFrame: RFM skoorid ja segmendid iga kliendi kohta
    """
    if reference_date is None:
        reference_date = pd.to_datetime('today')
    else:
        reference_date = pd.to_datetime(reference_date)

    df['sale_date'] = pd.to_datetime(df['sale_date'])

    # Recency: päevi viimasest ostust
    recency = df.groupby('customer_id')['sale_date'].max().reset_index()
    recency.columns = ['customer_id', 'last_purchase']
    recency['recency_days'] = (reference_date - recency['last_purchase']).dt.days

    # Frequency: ostude arv
    frequency = df.groupby('customer_id').size().reset_index(name='frequency')

    # Monetary: kogukulutus
    monetary = df.groupby('customer_id')['total_price'].sum().reset_index()  # Täidetud!
    monetary.columns = ['customer_id', 'monetary']

    # Liida kokku
    rfm = recency[['customer_id', 'recency_days']].merge(
        frequency, on='customer_id'
    ).merge(
        monetary, on='customer_id'
    )

    # Skooride määramine (lihtsustatud)
    rfm['R_score'] = pd.qcut(rfm['recency_days'], q=3, labels=[3, 2, 1]).astype(int)
    rfm['F_score'] = pd.qcut(
        rfm['frequency'].rank(method='first'), q=3, labels=[1, 2, 3]
    ).astype(int)
    rfm['M_score'] = pd.qcut(rfm['monetary'], q=3, labels=[1, 2, 3]).astype(int)
    rfm['RFM_score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

    # Segmenteerimine
    def assign_segment(score):
        if score >= 8:
            return 'Harjutuse tippkliendid'  # Täidetud!
        elif score >= 6:
            return 'Harjutuse püsikliendid'  # Täidetud!
        elif score >= 4:
            return 'Harjutuse kasvupotentsiaal'  # Täidetud!
        else:
            return 'Harjutuse riskikliendid'  # Täidetud!

    rfm['segment'] = rfm['RFM_score'].apply(assign_segment)
    return rfm

# Testi
rfm_result = calculate_rfm(df_all_orders, reference_date='2024-08-01')
print(rfm_result.sort_values('RFM_score', ascending=False))
print(f"\nSegmentide jaotus:")
print(rfm_result['segment'].value_counts())

      customer_id  recency_days  frequency  monetary  R_score  F_score  \
20         2026.0          -128          7   1933.66        3        3   
5          2008.0          -135          5   1294.32        3        3   
19         2025.0          -141          6   1804.79        3        3   
2545       4995.0          -208          4   1527.95        3        3   
2526       4973.0          -124          4   1253.86        3        3   
...           ...           ...        ...       ...      ...      ...   
1232       3455.0            46          1    351.62        1        1   
1280       3513.0            78          1    246.97        1        1   
443        2520.0           246          1     81.48        1        1   
2235       4632.0            33          1    164.43        1        1   
1266       3496.0            96          2    418.10        1        1   

      M_score  RFM_score                  segment  
20          3          9   Harjutuse tippkliendid  
5      

In [95]:
# Harjutus 2C
def city_comparison(df, cities=['Tallinn', 'Tartu', 'Pärnu']):
    """Võrdleb tellimuste arvu ja kogukäivet valitud linnade lõikes.

    Args:
        df: DataFrame tellimustega
        cities: Loend linnadest, mida võrrelda (vaikimisi kolm linna)

    Returns:
        dict: Linnade statistikakoondis
    """
    results = {}
    for city in cities:
        city_data = df[df['store_location'] == city]
        results[city] = {
            'orders': len(city_data),
            'revenue': round(city_data['total_price'].sum(), 2)
        }
    return results

# Test 1: Vaikimisi linnad
print("Test 1:", city_comparison(df_all_orders))

# Test 2: Ainult Tallinn ja Pärnu
print("Test 2:", city_comparison(df_all_orders, cities=['Tallinn', 'Pärnu']))

Test 1: {'Tallinn': {'orders': 3801, 'revenue': np.float64(1092083.15)}, 'Tartu': {'orders': 1797, 'revenue': np.float64(521603.11)}, 'Pärnu': {'orders': 1058, 'revenue': np.float64(288744.04)}}
Test 2: {'Tallinn': {'orders': 3801, 'revenue': np.float64(1092083.15)}, 'Pärnu': {'orders': 1058, 'revenue': np.float64(288744.04)}}


In [96]:
# Pipeline struktuur koodis
# === EXTRACT ===
def extract_data(supabase_client):
    """Too andmed Supabase'ist."""
    orders = fetch_all_orders(supabase_client, 'sales')
    customers = fetch_all_orders(supabase_client, 'customers')
    return orders, customers

# === TRANSFORM ===
def transform_data(orders, customers):
    """Puhasta ja analüüsi andmeid."""
    df = pd.merge(orders, customers, on='customer_id', how='left')
    df['sale_date'] = pd.to_datetime(df['sale_date'])
    rfm = calculate_rfm(df)  # Osa 2 funktsioon!
    return df, rfm

# === LOAD ===
def load_results(rfm, output_dir='reports'):
    """Salvesta tulemused."""
    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    rfm.to_csv(f'{output_dir}/rfm_{timestamp}.csv', index=False)
    px.bar(rfm['segment'].value_counts().reset_index(), x='segment', y='count',
           title=f'RFM Segmendid ({timestamp})').write_html(f'{output_dir}/rfm_{timestamp}.html')

In [97]:
# pipeline valideerimine
def validate(df):
    """Kontrolli andmete kvaliteeti."""
    if len(df) == 0:
        logger.warning("VALIDATE: Tühi andmestik!")
        return False
    return True

def run_pipeline():
    """Käivita kogu RFM pipeline."""
    try:
        orders, customers = extract_data(supabase)
        if not validate(orders):
            return False
        df, rfm = transform_data(orders, customers)
        load_results(rfm)
        return True
    except Exception as e:
        logger.error(f"PIPELINE FAILED: {e}")
        return False


In [98]:
# Pipline kontroll andmetega
import pandas as pd
import plotly.express as px
from datetime import datetime

# === EXTRACT ===
def extract_orders():
    """Simuleeri andmete toomist API-st."""
    print("[EXTRACT] Laadin...")
    data = {'customer_id': [1001,1002,1003,1001,1002,1004,1003,1001,1005,1004,
                            1002,1003,1005,1001,1006,1004,1002,1007,1003,1005],
            'sale_date': pd.date_range('2024-01-15', periods=20, freq='10D'),
            'total_price': [89.99,45.50,120.00,67.30,55.00,210.00,33.50,145.00,
                            78.00,92.00,160.00,44.00,88.50,230.00,37.00,175.00,
                            110.00,65.00,95.00,125.00],
            'store_location': ['Tallinn','Tartu','Tallinn','Tallinn','Tartu','Parnu','Tallinn',
                     'Tallinn','Tartu','Parnu','Tartu','Tallinn','Tartu','Tallinn',
                     'Parnu','Parnu','Tartu','Tallinn','Tallinn','Tartu']}
    df = pd.DataFrame(data)
    print(f"[EXTRACT] {len(df)} tellimust laaditud")
    return df

# === TRANSFORM ===
def transform_monthly(df):
    """Arvuta kuuraport."""
    print("[TRANSFORM] Arvutan...")
    monthly = df.groupby(df['sale_date'].dt.to_period('M')).agg(
        tellimusi=('sale_date', 'count'), kaive=('total_price', 'sum')
    ).reset_index()
    monthly['sale_date'] = monthly['sale_date'].astype(str)
    monthly['kaive'] = monthly['kaive'].round(2)
    print(f"[TRANSFORM] {len(monthly)} kuud")
    return monthly

# === LOAD ===
def load_report(monthly):
    """Salvesta CSV ja graafik."""
    ts = datetime.now().strftime('%Y%m%d_%H%M')
    monthly.to_csv(f'monthly_report_{ts}.csv', index=False)
    px.bar(monthly, x='sale_date', y='kaive',
           title=f'UrbanStyle kuukäive ({ts})',
           labels={'sale_date': 'Kuu', 'kaive': 'Käive (EUR)'}
    ).write_html(f'monthly_chart_{ts}.html')
    print(f"[LOAD] CSV + HTML salvestatud")

# === RUN ===
print("PIPELINE START")
df = extract_orders()
monthly = transform_monthly(df)
load_report(monthly)
print("PIPELINE COMPLETE")
print(monthly.to_string(index=False))

PIPELINE START
[EXTRACT] Laadin...
[EXTRACT] 20 tellimust laaditud
[TRANSFORM] Arvutan...
[TRANSFORM] 7 kuud
[LOAD] CSV + HTML salvestatud
PIPELINE COMPLETE
sale_date  tellimusi  kaive
  2024-01          2 135.49
  2024-02          3 242.30
  2024-03          3 388.50
  2024-04          3 330.00
  2024-05          3 362.50
  2024-06          3 322.00
  2024-07          3 285.00


In [99]:
# === TRANSFORM-RFM === Kontroll
def transform_rfm(df, reference_date='2024-08-01'):
    """Arvuta RFM segmendid."""
    print("[TRANSFORM-RFM] Arvutan...")
    ref = pd.to_datetime(reference_date)
    rfm = df.groupby('customer_id').agg(
        last_purchase=('sale_date', 'max'),
        frequency=('sale_date', 'count'),
        monetary=('total_price', 'sum')   # Täidetud: 'sum'
    ).reset_index()
    rfm['recency_days'] = (ref - rfm['last_purchase']).dt.days
    
    rfm['segment'] = rfm.apply(
        lambda row: 'Tippklient' if row['monetary'] > 200 and row['frequency'] >= 3
                    else 'Püsiklient' if row['frequency'] >= 3
                    else 'Ohus' if row['recency_days'] > 120
                    else 'Tavaklient', axis=1)
    return rfm

# Käivita laiendatud pipeline
df = extract_orders()
monthly = transform_monthly(df)
rfm = transform_rfm(df)
rfm

[EXTRACT] Laadin...
[EXTRACT] 20 tellimust laaditud
[TRANSFORM] Arvutan...
[TRANSFORM] 7 kuud
[TRANSFORM-RFM] Arvutan...


,customer_id,last_purchase,frequency,monetary,recency_days,segment
0,1001,2024-05-24,4,532.29,69,Tippklient
1,1002,2024-06-23,4,370.50,39,Tippklient
2,1003,2024-07-13,4,292.50,19,Tippklient
3,1004,2024-06-13,3,477.00,49,Tippklient
4,1005,2024-07-23,3,291.50,9,Tippklient
5,1006,2024-06-03,1,37.00,59,Tavaklient
6,1007,2024-07-03,1,65.00,29,Tavaklient


In [101]:
# Integreeriv harjutus — Marko täispipeline Urbanstyle andmetega
# Ülesanne: Ehita lõplik pipeline, mis ühendab API (või simuleeritud) andmete toomise, RFM arvutamise, kuuraportite loomise ja visualiseerimise.
import pandas as pd
import plotly.express as px
from datetime import datetime

# ============================================================
# MARKO IGANÄDALANE RFM PIPELINE
# ============================================================

# --- EXTRACT ---
def extract():
    """Too andmed (asenda Supabase API-ga, kui ühendus olemas)."""
    print(f"[EXTRACT] Alustan...")
    orders = pd.DataFrame({
        'customer_id': [1001,1002,1003,1001,1002,1004,1003,1001,1005,1004,
                        1002,1003,1005,1001,1006,1004,1002,1007,1003,1005],
        'sale_date': pd.date_range('2024-01-15', periods=20, freq='10D'),
        'total_price': [89.99,45.50,120.00,67.30,55.00,210.00,33.50,145.00,
                        78.00,92.00,160.00,44.00,88.50,230.00,37.00,175.00,
                        110.00,65.00,95.00,125.00],
        'store_location': ['Tallinn','Tartu','Tallinn','Tallinn','Tartu','Parnu',
                 'Tallinn','Tallinn','Tartu','Parnu','Tartu','Tallinn',
                 'Tartu','Tallinn','Parnu','Parnu','Tartu','Tallinn',
                 'Tallinn','Tartu']
    })
    customers = pd.DataFrame({
        'customer_id': [1001,1002,1003,1004,1005,1006,1007],
        'first_name': ['Juri','Kati','Maris','Peeter','Liina','Andres','Tiina'],
        'last_name': ['Tamm','Kask','Sepp','Rebane','Ots','Puu','Kuusk']
    })
    print(f"[EXTRACT] {len(orders)} tellimust, {len(customers)} klienti")
    return orders, customers

# --- TRANSFORM ---
def transform(orders, customers, reference_date='2024-08-01'):
    """Puhasta, arvuta RFM, loo kuuraport."""
    print(f"[TRANSFORM] Alustan...")
    ref = pd.to_datetime(reference_date)
    df = pd.merge(orders, customers, on='customer_id', how='left')

    # Kuuraport
    monthly = df.groupby(df['sale_date'].dt.to_period('M')).agg(
        tellimusi=('sale_date', 'count'), kaive=('total_price', 'sum')
    ).reset_index()
    monthly['sale_date'] = monthly['sale_date'].astype(str)

    # R, F, M väärtused + lihtsustatud reeglisegmendid (mitte N7 R+F+M skoor)
    rfm = df.groupby('customer_id').agg(
        last_purchase=('sale_date', 'max'), frequency=('sale_date', 'count'),
        monetary=('total_price', 'sum'), nimi=('first_name', 'first')
    ).reset_index()
    rfm['recency_days'] = (ref - rfm['last_purchase']).dt.days
    # 300 € = lihtsustatud harjutuse lävend – ei ole N7 RFM ega suurklient (>1000 €)
    rfm['segment'] = rfm.apply(
        lambda r: 'Tippklient' if r['monetary'] > 300 and r['frequency'] >= 3
                  else 'Püsiklient' if r['frequency'] >= 3
                  else 'Ohus' if r['recency_days'] > 120
                  else 'Tavaklient', axis=1)

    print(f"[TRANSFORM] {len(rfm)} klienti segmenteeritud")
    return {'monthly': monthly, 'rfm': rfm}

# --- VALIDATE ---
def validate(results):
    """Kontrolli andmete kvaliteeti."""
    ok = len(results['rfm']) > 0 and results['monthly']['kaive'].sum() > 0
    print(f"[VALIDATE] {'OK' if ok else 'PROBLEEM!'}")
    return ok

# --- LOAD ---
def load(results):
    """Salvesta CSV ja graafikud."""
    ts = datetime.now().strftime('%Y%m%d')
    results['rfm'].to_csv(f'rfm_report_{ts}.csv', index=False)
    px.scatter(results['rfm'], x='recency_days', y='monetary', color='segment',
               size='frequency', hover_data=['nimi'],
               title=f'UrbanStyle RFM ({ts})',
               labels={'recency_days': 'Paevi viimasest ostust', 'monetary': 'Kogukulutus (EUR)'}
    ).write_html(f'rfm_chart_{ts}.html')
    px.bar(results['monthly'], x='sale_date', y='kaive',
           title=f'Kuukäive ({ts})', labels={'sale_date': 'Kuu', 'kaive': 'EUR'}
    ).write_html(f'monthly_chart_{ts}.html')
    print(f"[LOAD] 1 CSV + 2 HTML salvestatud")

# --- RUN ---
print("=" * 50)
print("  MARKO IGANADALANE RFM PIPELINE")
print("=" * 50)
start = datetime.now()
orders, customers = extract()
results = transform(orders, customers)
if validate(results):
    load(results)
print(f"  VALMIS ({(datetime.now() - start).total_seconds():.1f}s)")
print("=" * 50)

# Kokkuvõte
print("\n--- SEGMENDID ---")
print(results['rfm'][['nimi', 'frequency', 'monetary', 'segment']]
      .sort_values('monetary', ascending=False).to_string(index=False))

  MARKO IGANADALANE RFM PIPELINE
[EXTRACT] Alustan...
[EXTRACT] 20 tellimust, 7 klienti
[TRANSFORM] Alustan...
[TRANSFORM] 7 klienti segmenteeritud
[VALIDATE] OK
[LOAD] 1 CSV + 2 HTML salvestatud
  VALMIS (0.2s)

--- SEGMENDID ---
  nimi  frequency  monetary    segment
  Juri          4    532.29 Tippklient
Peeter          3    477.00 Tippklient
  Kati          4    370.50 Tippklient
 Maris          4    292.50 Püsiklient
 Liina          3    291.50 Püsiklient
 Tiina          1     65.00 Tavaklient
Andres          1     37.00 Tavaklient


In [ ]:
# Lisaharjutus 10: Kirjuta Python funktsioon total_revenue(df), mis võtab sisendiks DataFrame'i (veerg total_price) ja tagastab kogukäive ümardatuna 2 kohani. Lisa docstring ja vigade käsitlemine.
import pandas as pd

def total_revenue(df):
    """
    Arvutab DataFrame'i veeru 'total_price' põhjal kogukäibe
    ja ümardab tulemuse kahe komakohani.
    """
    try:
        if 'total_price' not in df.columns:
            raise ValueError("Veerg 'total_price' puudub andmestikust.")
            
        revenue = df['total_price'].sum()
        return round(float(revenue), 2)
        
    except Exception as e:
        print(f"[VIGA] Kogukäibe arvutamine ebaonnestus: {e}")
        return None

    # Kutsu funktsioon välja ja prindi tulemus:
tulemus = total_revenue(df)
print("Kogukäive:", tulemus)

Kogukäive: 2065.79
